In [12]:
import os, io, numpy as np
import tensorflow as tf
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, HTML

model = tf.keras.applications.R(weights= 'imagenet' )

display(HTML("""
<style>
    .vision-container {background: #020617; border: 2px solid #38bdf8; border-radius: 20px; padding: 30px; font-family: 'Segoe UI' , Tahoma, Geneva, Verdana, sans-serif; text-align: center; box-shadow: 0 0 50px rgba(56, 189, 248, 0.2); }
    .header-title { color: #38bdf8; font-size: 32px; font-weight: 900; letter-spacing: 5px; text-shadow: 0 0 15px #38bdf8; margin-bottom: 5px; }
    .header-subtitle { color: #94a3b8; font-size: 14px; margin-bottom: 25px; text-transform: uppercase; }
    .stats-box { display: flex; justify-content: space-around; margin: 20px 0; }
    .stat-item { background: #0f172a; border: 1px solid #1e293b; padding: 15px; border-radius: 12px; width: 30%; }
    .stat-value { color: #38bdf8; font-size: 18px; font-weight: bold; display: block; }
    .stat-label { color: #64748b; font-size: 11px; text-transform: uppercase; }
    .upload-section { border: 2px dashed #1e293b; border-radius: 15px; padding: 40px; margin: 20px 0; transition: 0.3s; }
    .upload-section:hover { border-color: #38bdf8; background: #0f172a; }
    .result-card { background: linear-gradient(145deg, #1e293b, #0f172a); border-radius: 15px; padding: 20px; margin-top: 20px; text-align: left; border-right: 4px solid #38bdf8; }
    .weight-tag { background: #38bdf8; color: #020617; padding: 2px 8px; border-radius: 4px; font-size: 12px; font-weight: bold; }
</style>

<div class="vision-container">
    <div class="header-title">VISION SYSTEM 2030</div>
    <div class="header-subtitle">Neural Engine & Object Recognition Unit</div>
    <div class="stats-box">
        <div class="stat-item"><span class="stat-value">3.12.1</span><span class="stat-label">Python Kernel</span></div>
        <div class="stat-item"><span class="stat-value">Active</span><span class="stat-label">GPU/CPU Ops</span></div>
        <div class="stat-item"><span class="stat-value">MobileNetV2</span><span class="stat-label">AI Model</span></div>
    </div>
</div>
"""))

output = widgets.Output()
uploader = widgets.FileUpload(accept= 'image/*', multiple=False, description='إطلاق الماسح الضوئي')
uploader.style.button_color =  '#38bdf8'

def process_vision(change):
    with output:
        output.clear_output()
        if not uploader.value: return
        
        file_data = list(uploader.value)[0]
        img = Image.open(io.BytesIO(file_data[ 'content' ])).convert( 'RGB' )
        img_input = img.resize((224, 224))
        
        display(HTML("<div style= 'text-align:center; padding:20px;' ><h3 style= 'color:#38bdf8;' >جاري فحص الطبقات العصبية...</h3></div>"))
        display(img.resize((400, 300)))
        
        x = tf.keras.preprocessing.image.img_to_array(img_input)
        x = np.expand_dims(x, axis=0)
        x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
        
        preds = model.predict(x)
        decoded = tf.keras.applications.mobilenet_v2.decode_predictions(preds, top=3)[0]
        
        weights_sample = model.layers[1].get_weights()[0][0][0][0][:4]
        
        res_html = "<div class= 'result-card' >"
        res_html += f"<h2 style= 'color:#38bdf8; margin-top:0'; >تقرير التحليل البصري:</h2>"
        for i, (id, label, prob) in enumerate(decoded):
            res_html += f"<p style= 'color:white; font-size:16px;' >{i+1}. <b>{label.upper()}</b> - <span style= 'color:#4ade80;' >Confidence: {prob*100:.2f}%</span></p>"
        
        res_html += f"<hr style=' border:0; border-top:1px solid #1e293b;' >"
        res_html += f"<p style=' color:#94a3b8;' ><b>Neural Weights (Layer_1):</b> <span class= 'weight-tag' >{list(np.round(weights_sample, 4))}</span></p>"
        res_html += "</div>"
        display(HTML(res_html))

uploader.observe(process_vision, names= 'value' )
display(widgets.VBox([uploader, output], layout=widgets.Layout(align_items= 'center' , margin= '20px' )))
